# Phase 3: Machine Learning Models

**Days 19-27**: Forecasting, Anomaly Detection, Risk Scoring

In [ ]:
import pandas as pd
import numpy as np
import sys
from sqlalchemy import create_engine

sys.path.insert(0, '../src')

from pfm.config import DB_PATH, USERS
from pfm.models.forecaster import prepare_forecast_data, train_linear_regression, train_random_forest, train_xgboost
from pfm.models.anomaly_detector import detect_anomalies_isolation_forest, detect_anomalies_zscore
from pfm.models.risk_scorer import financial_health_score, risk_categories

engine = create_engine(f'sqlite:///{DB_PATH}')
print('✅ ML Models Ready')

In [ ]:
df = pd.read_sql('SELECT * FROM transactions', engine)
df['date'] = pd.to_datetime(df['date'])
budgets = pd.read_sql('SELECT * FROM budgets', engine)

print(f'Data loaded: {len(df):,} transactions')

## Expense Forecasting

In [ ]:
print('\n' + '='*80)
print('📊 EXPENSE FORECASTING MODELS')
print('='*80)

for user_info in USERS[:1]:  # Demo with first user
    user_id = user_info['user_id']
    user_name = user_info['user_name']
    
    print(f'\n🔮 {user_name}: Training Models...')
    
    try:
        X, y = prepare_forecast_data(df, user_id)
        
        # Train models
        lr_result = train_linear_regression(X, y)
        rf_result = train_random_forest(X, y)
        xgb_result = train_xgboost(X, y)
        
        print(f'\n  Linear Regression: MAE={lr_result["mae"]}, R²={lr_result["r2"]}')
        print(f'  Random Forest:     MAE={rf_result["mae"]}, R²={rf_result["r2"]}')
        print(f'  XGBoost:           MAE={xgb_result["mae"]}, R²={xgb_result["r2"]}')
        
        best_model = max([lr_result, rf_result, xgb_result], key=lambda x: x['r2'])
        print(f'\n  ✅ Best Model: {best_model["type"]} (R²={best_model["r2"]})')
    except Exception as e:
        print(f'  ⚠️ Error: {e}')

## Anomaly Detection

In [ ]:
print('\n' + '='*80)
print('🚨 ANOMALY DETECTION')
print('='*80)

for user_info in USERS:
    user_id = user_info['user_id']
    user_name = user_info['user_name']
    
    # Isolation Forest
    if_result = detect_anomalies_isolation_forest(df, user_id)
    
    print(f'\n{user_name}')
    print(f'  Isolation Forest: {if_result["anomaly_count"]} anomalies ({if_result["anomaly_percentage"]}%)')
    if len(if_result['anomalies']) > 0:
        print(f'  Avg anomaly amount: Rs. {if_result["mean_anomaly_amount"]:,.0f}')

## Financial Health Score

In [ ]:
print('\n' + '='*80)
print('💯 FINANCIAL HEALTH SCORES')
print('='*80)

for user_info in USERS:
    user_id = user_info['user_id']
    user_name = user_info['user_name']
    
    score_result = financial_health_score(df, budgets, user_id)
    
    print(f'\n{user_name}')
    print(f'  Overall Score: {score_result["overall_score"]}/100 - {score_result["rating"]}')
    print(f'  Breakdown: Savings={score_result["score_breakdown"]["savings"]}, Budget={score_result["score_breakdown"]["budget"]}, Debt={score_result["score_breakdown"]["debt"]}')

## Risk Categories

In [ ]:
print('\n' + '='*80)
print('⚠️ AT-RISK SPENDING CATEGORIES')
print('='*80)

for user_info in USERS:
    user_id = user_info['user_id']
    user_name = user_info['user_name']
    
    risk_result = risk_categories(df, budgets, user_id)
    
    print(f'\n{user_name}: {risk_result["total_at_risk_categories"]} categories at risk')
    for cat, details in risk_result['at_risk_details'].items():
        print(f'  {cat}: {details["ratio"]}% of budget ({details["risk_level"]})')

In [ ]:
print('\n' + '='*80)
print('✅ Phase 3 Complete: ML Models Trained')
print('='*80)